In [1]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, models

In [2]:
seed = 42
tf.random.set_seed(seed)
np.random.seed(seed)

In [3]:
from pathlib import Path
train_ds, val_ds = tf.keras.utils.audio_dataset_from_directory(
    directory=str(Path.home() / ".cache" / "ml-notebooks" / "speech-commands"),
    batch_size=64,
    output_sequence_length=16000,
    seed=seed,
    validation_split=0.3,
    subset='both'
)

Found 6505 files belonging to 4 classes.
Using 4554 files for training.
Using 1951 files for validation.


In [4]:
label = np.array(train_ds.class_names)
print(label)

['follow' 'learn' 'tree' 'visual']


In [5]:
def squeeze(audio, label):
    audio = tf.squeeze(audio, axis=-1)
    return audio, label

def get_spectogram(waveform):
    spectogram = tf.signal.stft(waveform, frame_length=255, frame_step=128)
    spectogram = tf.abs(spectogram)
    spectogram = spectogram[..., tf.newaxis]
    return spectogram

def make_spectogram_ds(ds):
    return ds.map(
        lambda audio, label: (get_spectogram(audio), label),
        tf.data.AUTOTUNE
    )

In [6]:
train_ds = train_ds.map(squeeze, tf.data.AUTOTUNE)
val_ds = val_ds.map(squeeze, tf.data.AUTOTUNE)

test_ds = val_ds.shard(num_shards=2, index=0)
val_ds = val_ds.shard(num_shards=2, index=1)

In [7]:
train_spectogram_ds = make_spectogram_ds(train_ds)
val_spectogram_ds = make_spectogram_ds(val_ds)
test_spectogram_ds = make_spectogram_ds(test_ds)

In [10]:
for i, j in train_spectogram_ds.take(1):
    break

input_shape = i.shape[1:]
num_label = len(label)

In [11]:
norm_layer = layers.Normalization()
norm_layer.adapt(
    data=train_spectogram_ds.map(lambda i, j: i)
)

In [12]:
model = models.Sequential([
    layers.Input(shape=input_shape),
    layers.Resizing(32, 32),
    norm_layer,
    layers.Conv2D(64, 3, activation='relu'),
    layers.MaxPooling2D(),
    layers.Conv2D(128, 3, activation='relu'),
    layers.MaxPooling2D(),
    layers.Dropout(0.5),
    layers.Flatten(),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(num_label)
])

In [13]:
model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 resizing (Resizing)         (None, 32, 32, 1)         0         
                                                                 
 normalization (Normalizatio  (None, 32, 32, 1)        3         
 n)                                                              
                                                                 
 conv2d (Conv2D)             (None, 30, 30, 64)        640       
                                                                 
 max_pooling2d (MaxPooling2D  (None, 15, 15, 64)       0         
 )                                                               
                                                                 
 conv2d_1 (Conv2D)           (None, 13, 13, 128)       73856     
                                                                 
 max_pooling2d_1 (MaxPooling  (None, 6, 6, 128)        0

In [14]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(),
    loss=tf.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=['accuracy']
)

In [15]:
history = model.fit(
    train_spectogram_ds,
    validation_data=val_spectogram_ds,
    epochs=8,
    callbacks=tf.keras.callbacks.EarlyStopping(verbose=1, patience=2)
)

Epoch 1/8
72/72 [==============================] - 5s 56ms/step - loss: 0.8565 - accuracy: 0.6647 - val_loss: 0.5337 - val_accuracy: 0.7937
Epoch 2/8
72/72 [==============================] - 3s 37ms/step - loss: 0.5199 - accuracy: 0.8206 - val_loss: 0.3679 - val_accuracy: 0.8635
Epoch 3/8
72/72 [==============================] - 3s 37ms/step - loss: 0.4000 - accuracy: 0.8617 - val_loss: 0.3068 - val_accuracy: 0.8781
Epoch 4/8
72/72 [==============================] - 3s 40ms/step - loss: 0.3492 - accuracy: 0.8779 - val_loss: 0.2796 - val_accuracy: 0.9010
Epoch 5/8
72/72 [==============================] - 3s 44ms/step - loss: 0.2923 - accuracy: 0.8994 - val_loss: 0.2307 - val_accuracy: 0.9198
Epoch 6/8
72/72 [==============================] - 3s 44ms/step - loss: 0.2400 - accuracy: 0.9181 - val_loss: 0.2149 - val_accuracy: 0.9292
Epoch 7/8
72/72 [==============================] - 3s 46ms/step - loss: 0.2245 - accuracy: 0.9240 - val_loss: 0.1856 - val_accuracy: 0.9323
Epoch 8/8
72/72 [===

In [17]:
for i, (spec_batch, lab_batch) in enumerate(test_spectogram_ds):
    pred = model.predict(spec_batch)

    for j in range(len(lab_batch)):
        t = lab_batch[j].numpy()
        p = np.argmax(pred[j])

        print(f'Batch {i}, Data {j}, ori -> {label[t]}, pred -> {label[p]}')

2/2 [==============================] - 0s 10ms/step
Batch 0, Data 0, ori -> tree, pred -> tree
Batch 0, Data 1, ori -> learn, pred -> learn
Batch 0, Data 2, ori -> tree, pred -> tree
Batch 0, Data 3, ori -> follow, pred -> follow
Batch 0, Data 4, ori -> tree, pred -> tree
Batch 0, Data 5, ori -> follow, pred -> follow
Batch 0, Data 6, ori -> follow, pred -> follow
Batch 0, Data 7, ori -> follow, pred -> follow
Batch 0, Data 8, ori -> follow, pred -> follow
Batch 0, Data 9, ori -> learn, pred -> learn
Batch 0, Data 10, ori -> visual, pred -> visual
Batch 0, Data 11, ori -> learn, pred -> learn
Batch 0, Data 12, ori -> tree, pred -> tree
Batch 0, Data 13, ori -> follow, pred -> follow
Batch 0, Data 14, ori -> follow, pred -> learn
Batch 0, Data 15, ori -> tree, pred -> tree
Batch 0, Data 16, ori -> learn, pred -> learn
Batch 0, Data 17, ori -> follow, pred -> follow
Batch 0, Data 18, ori -> visual, pred -> visual
Batch 0, Data 19, ori -> learn, pred -> learn
Batch 0, Data 20, ori -> lear